# Autoresearch Car-Price Model — Experiment Analysis

Analysis of the autonomous `val_mae` tuning loop from `results.tsv`.
Lower MAE (euros) is better; the running minimum is the "frontier".

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_mae, val_mape, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_mae"] = pd.to_numeric(df["val_mae"], errors="coerce")
df["val_mape"] = pd.to_numeric(df["val_mape"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    mae = row["val_mae"]
    print(f"  #{i:3d}  val_mae=EUR {mae:,.0f}  MAPE={row['val_mape']:.1f}%  {row['description']}")

## Val MAE Over Time

Track how the best (kept) `val_mae` evolves as experiments progress. The running
minimum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_mae = valid.loc[0, "val_mae"]

# Only plot points within a small margin above baseline (the interesting region)
below = valid[valid["val_mae"] <= baseline_mae * 1.02]

# Plot discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_mae"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_mae"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_mae = valid.loc[kept_mask, "val_mae"]
running_min = kept_mae.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, mae in zip(kept_idx, kept_mae):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, mae),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation MAE (EUR, lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below best to just above baseline (guard against a single row)
best_mae = kept_mae.min()
margin = max((baseline_mae - best_mae) * 0.15, baseline_mae * 0.01)
ax.set_ylim(best_mae - margin, baseline_mae + margin)

plt.tight_layout()
plt.savefig("autoresearch-progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to autoresearch-progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_mae = df.iloc[0]["val_mae"]
best_mae = kept["val_mae"].min()
best_row = kept.loc[kept["val_mae"].idxmin()]

print(f"Baseline val_mae:  EUR {baseline_mae:,.2f}")
print(f"Best val_mae:      EUR {best_mae:,.2f}")
print(f"Total improvement: EUR {baseline_mae - best_mae:,.2f} "
      f"({(baseline_mae - best_mae) / baseline_mae * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    print(f"  Experiment #{row['index']:3d}: val_mae=EUR {row['val_mae']:,.2f}  {str(row['description']).strip()}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's val_mae
# (experiments are cumulative -- each builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_mae"] = kept["val_mae"].shift(1)
kept["delta"] = kept["prev_mae"] - kept["val_mae"]

hits = kept.iloc[1:].copy()              # drop baseline (no delta)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta(EUR)':>11}  {'val_mae':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+11.2f}  {row['val_mae']:10.2f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+11.2f}  {'':>10}  TOTAL improvement over baseline")